**Requerimientos**


In [0]:
%pip install pyyaml

**Cargando config.yml**

In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit, col, to_timestamp
import yaml

with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)
print("✅ Configuración cargada")

**Creando esquema silver**

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['spark']['database']}.gold")
print(f"✅ Esquema {config['spark']['database']}.gold creado")

**Leyendo tabla bronze**

In [0]:
silver_df = spark.table(config["tables"]["silver_table"])
display(silver_df.limit(5))
print("✅ Datatable cargada")

In [0]:
from pyspark.sql.functions import (
    col, month, year, avg, max, min, count,
    round as spark_round, when, sum as spark_sum
)

# ── Agrupación principal ─────────────────────────────────────────────────────
gold_df = silver_df \
    .withColumn("year",  year(col("Date_Time"))) \
    .withColumn("month", month(col("Date_Time"))) \
    .groupBy("Location", "year", "month") \
    .agg(
        spark_round(avg("Temperature_C"),    2).alias("avg_temp_c"),
        spark_round(max("Temperature_C"),    2).alias("max_temp_c"),
        spark_round(min("Temperature_C"),    2).alias("min_temp_c"),
        spark_round(avg("Humidity_pct"),     2).alias("avg_humidity_pct"),
        spark_round(avg("Precipitation_mm"), 2).alias("avg_precipitation_mm"),
        spark_round(avg("Wind_Speed_kmh"),   2).alias("avg_wind_kmh"),
        count("*").alias("record_count"),

        # Conteo de días con clima extremo
        spark_sum(when(col("Temperature_C")    >  35, 1).otherwise(0)).alias("extreme_heat_days"),
        spark_sum(when(col("Temperature_C")    < -10, 1).otherwise(0)).alias("extreme_cold_days"),
        spark_sum(when(col("Humidity_pct")     >  90, 1).otherwise(0)).alias("high_humidity_days"),
        spark_sum(when(col("Precipitation_mm") >   8, 1).otherwise(0)).alias("heavy_rain_days"),
        spark_sum(when(col("Wind_Speed_kmh")   >  25, 1).otherwise(0)).alias("strong_wind_days"),
    ) \
    .withColumn("total_extreme_days",
        col("extreme_heat_days") + col("extreme_cold_days") +
        col("high_humidity_days") + col("heavy_rain_days") +
        col("strong_wind_days")
    ) \
    .orderBy("Location", "year", "month")

display(gold_df.limit(5))

In [0]:
gold_df.write.format("delta") \
    .mode(config["spark"]["write_mode"]) \
    .saveAsTable(config["tables"]["gold_table"])

print(f"✅ Dataset guardado en {config['tables']['gold_table']}")
display(spark.sql(f"DESCRIBE DETAIL {config['tables']['gold_table']}"))